In [1]:
import sys, os
os.chdir('..')
sys.path.append('scripts')
os.environ['OPENAI_API_KEY']='REPLACED_BY_ENV_LOADER_SEE_README'

In [2]:
from video_segmentation import compute_segment_visual_changes, segment_enhanced
from llm_segmentation import verify_cuts_with_llm, segment_with_llm, evaluate_segmentation
import json, numpy as np

In [3]:
VIDEO = "UltrasoundCrawler_KeyCode_20260323_v2/output/20260520_162816_youtube/media/case_reasoning/8V649L5Q368.mp4"
with open("transcripts/8V649L5Q368.json") as f:
    asr_data = json.load(f)
segments = asr_data["segments"]

for segment in segments[:5]:
    print(segment)

{'start': 0.7, 'end': 4.46, 'text': "Hi, I'm Dr. John Kugler with the Stanford 25 Ultra Sound series."}
{'start': 4.46, 'end': 7.08, 'text': "Today, we're going to be learning about lung ultrasound."}
{'start': 7.08, 'end': 11.98, 'text': 'Now, lung ultrasound is really important, especially in my practice as a hospitalist.'}
{'start': 11.98, 'end': 15.98, 'text': 'I use lung ultrasound all the time to evaluate patients for various conditions.'}
{'start': 16.22, 'end': 22.31, 'text': 'Today, I want to focus on pneumothorax, pleural effusions, as well as telling the'}


In [4]:
visual_changes = compute_segment_visual_changes(VIDEO, segments, threshold=0.3)

## Histogram + LLM


In [5]:
llm_result = verify_cuts_with_llm(segments, visual_changes)
clips_b = segment_with_llm(segments, llm_result, min_clip=30)
print(f"\nMethod B: {len(clips_b)} clips")
for c in clips_b:
    print(f"  {c['start']:.0f}-{c['end']:.0f}s ({c['duration']:.0f}s) | {c.get('topic','')[:40]}")

  Calling GPT-4o for cut verification...
  LLM response: 6.9s | 8754 tokens

Method B: 10 clips
  28-87s (59s) | Introduction and overview of lung ultras
  87-250s (163s) | Diagnosis of pneumothorax using ultrasou
  250-323s (73s) | Diagnosis of right-sided pleural effusio
  323-399s (75s) | Diagnosis of pleural effusions using ult
  399-444s (45s) | Diagnosis of left-sided pleural effusion
  444-551s (107s) | Challenges and techniques for identifyin
  551-622s (71s) | Introduction to lung ultrasound artifact
  622-771s (149s) | Techniques for lung ultrasound and ident
  771-983s (212s) | Interpreting lung ultrasound findings an
  983-1126s (144s) | Clinical integration and experience in l


In [6]:
print(json.dumps(llm_result, indent=2, ensure_ascii=False))

{
  "final_cuts": [
    {
      "segment_index": 20,
      "time": 87.5,
      "should_cut": true,
      "topic": "Introduction and overview of lung ultrasound and pneumothorax"
    },
    {
      "segment_index": 52,
      "time": 250.2,
      "should_cut": true,
      "topic": "Diagnosis of pneumothorax using ultrasound"
    },
    {
      "segment_index": 80,
      "time": 398.8,
      "should_cut": true,
      "topic": "Diagnosis of pleural effusions using ultrasound"
    },
    {
      "segment_index": 111,
      "time": 551.3,
      "should_cut": true,
      "topic": "Challenges and techniques for identifying pleural effusions"
    },
    {
      "segment_index": 125,
      "time": 622.1,
      "should_cut": true,
      "topic": "Introduction to lung ultrasound artifacts: A-lines and B-lines"
    },
    {
      "segment_index": 154,
      "time": 770.7,
      "should_cut": true,
      "topic": "Techniques for lung ultrasound and identifying A-lines and B-lines"
    },
    {
     

In [7]:
from pathlib import Path
output = {
    'video_id': asr_data['video_id'],
    'video_path': VIDEO,
    'duration_sec': asr_data['duration_sec'],
    'method': 'histogram_llm',
    'num_clips': len(clips_b),
    'coverage_pct': round(sum(c['duration'] for c in clips_b) / asr_data['duration_sec'] * 100, 1),
    'clips': clips_b,
    'llm_result': llm_result,
}

Path('results').mkdir(exist_ok=True)
out_path = f"results/{asr_data['video_id']}_clips.json"
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f"Saved to {out_path}")

Saved to results/8V649L5Q368_clips.json
